# 06 — Preprocessing

**PLTMH–ELC–QLSTM**

Notebook ini menangani seluruh proses antara **trajectory temporal
mentah** dan **tensor supervised final** untuk LSTM dan QLSTM.

## Batas ilmiah

Input Notebook 06:

**RAW_LABELED_TEMPORAL_TRAJECTORIES**

dari `05_generate_dataset.ipynb`.

Output Notebook 06:

**FINAL_LSTM_QLSTM_READY**

yang terdiri atas:

- feature akhir;
- observasi 20 Hz;
- causal window;
- pembagian keluarga train/validation/test;
- scaler berbasis data training saja;
- tensor final;
- metadata sample dan split.

Notebook ini **tidak melakukan training LSTM maupun QLSTM**.


## 1. Protokol Preprocessing yang Dibekukan

Empat feature akhir adalah:

\[
x(t)=
[\Delta f(t),\;P_m(t),\;P_e(t),\;P_{dump}(t)]
\]

dengan:

- `frequency_deviation_hz`;
- `mechanical_power_kw`;
- `electrical_power_kw`;
- `dump_power_kw`.

Observation cadence:

\[
\Delta t_{obs}=0.05\;\text{s}=20\;\text{Hz}
\]

Panjang sequence:

\[
N_w=11
\]

sehingga window mempunyai rentang:

\[
(11-1)(0.05)=0.5\;\text{s}
\]

Sebuah window hanya eligible bila:

\[
t_{end} > t_d
\]

sehingga window dengan endpoint tepat pada atau sebelum gangguan tidak
digunakan sebagai sample supervised.

Split dilakukan pada **dynamic family level**, bukan secara acak pada
baris atau window. Scaler hanya boleh di-*fit* menggunakan bagian
training.


## 2. Mode Eksekusi

Notebook ini mempunyai dua mode.

### Mode verifikasi — default

```python
REBUILD_PREPROCESSING = False
```

Mode ini membaca dan memverifikasi artefak final yang telah dibekukan.
Tidak ada dataset ilmiah yang ditulis ulang.

### Mode reproduksi eksplisit

Untuk mereproduksi preprocessing dari trajectory Notebook 05:

```python
REBUILD_PREPROCESSING = True
```

File raw yang diperlukan:

`data/raw/notebook05_raw_temporal_trajectories.csv.gz`

Hasil reproduksi **tidak pernah menimpa `data/final/`**. Output
reproduksi ditulis ke `data/reproduction/` dan dibandingkan terhadap
kontrak ilmiah final.


In [ ]:
# ============================================================
# 06.1 — PROJECT SETUP + FROZEN ARTIFACT INTEGRITY
# ============================================================

from pathlib import Path

import hashlib
import json
import sys

import numpy as np
import pandas as pd


EXPECTED_UPSTREAM_CHECKPOINT = "9f6a764e80dfed06b577584e80caa577d4162ca3"

EXPECTED_NOTEBOOK_05_SHA256 = (
    "786ce9f7692b8d71bf3fe73629742f6475ae779e70f2a86df4b6420bfd811594"
)

EXPECTED_ARTIFACT_SHA256 = {
    "dataset": "7287018d944b7e3740aa778c78459df1142d53d77493a6193493af2009b46f60",
    "feature_scaler": "e302402cb371a1e0223051cd752e8d9de398651373bc229309c675a36bb96096",
    "metadata": "0e11a631937bbe349fe90bf326ba97fd5e14d209859ad10ca159b6fdf74efcf5",
    "sample_metadata": "9d3f5297dc788231abefebb0dbbd4e98880ef3a6e33a3af6033fe68963d9aa5c",
    "scalers": "7061abdbe1e8cfeec0886a2030bc9a661f75d74204953929c278bc49b4f73d17",
    "split_assignment": "63cea54cdcb1a9922640ab2cbd387642ff79b00ae731540bf119680858ac595f",
    "split_indices": "08b59436f2c108467d8db0258cf519ed2edc3a69faf25feacf4459b662cc38de",
    "split_metadata": "08dfda462f996fb9654338e6f7468808c2d7d834a9d821c705fa02de2c5e7b69",
    "split_structural_audit": "f22d30e1fbf3f004323408e53d35620c0174ffa1d5996495a67f89b85837bb50",
    "target_scaler": "9dbd52fe5f99a2aa47275f85edc802b775db5879e00dcb54a24124a9d2ef0ab6",
    "window_split_metadata": "e4c5a3db5fc14676e5cfc251f563d914700cb5bd25e37fda86be666f796bbf71"
}

ARTIFACT_RELPATHS = {
    "dataset": "data/final/qlstm_lstm_dataset_final.npz",
    "feature_scaler": "data/final/qlstm_lstm_feature_scaler.csv",
    "metadata": "data/final/qlstm_lstm_dataset_metadata.json",
    "sample_metadata": "data/final/qlstm_lstm_sample_metadata.csv.gz",
    "scalers": "data/final/qlstm_lstm_scalers.json",
    "split_assignment": "data/splits/cell137_family_split_assignment.csv",
    "split_indices": "data/splits/cell137_split_indices.npz",
    "split_metadata": "data/splits/cell137_split_metadata.json",
    "split_structural_audit": "data/splits/cell137_split_structural_audit.csv.gz",
    "target_scaler": "data/final/qlstm_lstm_target_scaler.csv",
    "window_split_metadata": "data/splits/cell137_window_split_metadata.csv.gz"
}

EXPECTED_ARRAY_SCHEMA = {
    "X_test": {
        "dtype": "float32",
        "shape": [
            3200,
            11,
            4
        ]
    },
    "X_train": {
        "dtype": "float32",
        "shape": [
            11200,
            11,
            4
        ]
    },
    "X_validation": {
        "dtype": "float32",
        "shape": [
            3200,
            11,
            4
        ]
    },
    "feature_names": {
        "dtype": "<U22",
        "shape": [
            4
        ]
    },
    "target_names": {
        "dtype": "<U9",
        "shape": [
            2
        ]
    },
    "test_index": {
        "dtype": "int64",
        "shape": [
            3200
        ]
    },
    "train_index": {
        "dtype": "int64",
        "shape": [
            11200
        ]
    },
    "validation_index": {
        "dtype": "int64",
        "shape": [
            3200
        ]
    },
    "y_test": {
        "dtype": "float32",
        "shape": [
            3200,
            2
        ]
    },
    "y_train": {
        "dtype": "float32",
        "shape": [
            11200,
            2
        ]
    },
    "y_validation": {
        "dtype": "float32",
        "shape": [
            3200,
            2
        ]
    }
}

CANONICAL_ARRAY_KEYS = {
    "X_test": "X_test",
    "X_train": "X_train",
    "X_val": "X_validation",
    "y_test": "y_test",
    "y_train": "y_train",
    "y_val": "y_validation"
}

EXPECTED_SAMPLE_METADATA_COLUMNS = [
    "window_id",
    "scenario_id",
    "family_key",
    "gain_regime",
    "mechanical_power_pu",
    "initial_dump_power_kw",
    "load_step_kw",
    "disturbance_time_s",
    "window_start_time_s",
    "window_end_time_s",
    "window_start_relative_to_disturbance_s",
    "window_end_relative_to_disturbance_s",
    "contains_pre_event_context",
    "contains_event_time",
    "kp_target",
    "ki_target",
    "split",
    "tensor_row_index"
]


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "src").exists()
        and
        (candidate / "notebooks").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


NOTEBOOK_05_PATH = (
    PROJECT_ROOT
    /
    "notebooks"
    /
    "05_generate_dataset.ipynb"
)


if (
    sha256_file(
        NOTEBOOK_05_PATH
    )
    !=
    EXPECTED_NOTEBOOK_05_SHA256
):

    raise RuntimeError(
        "Upstream Notebook 05 changed."
    )


ARTIFACT_PATHS = {
    name:
        PROJECT_ROOT / relative_path

    for name, relative_path
    in ARTIFACT_RELPATHS.items()
}


artifact_integrity = {}


for name, path in ARTIFACT_PATHS.items():

    if not path.exists():

        raise RuntimeError(
            f"Missing frozen artifact: {path}"
        )


    current_sha = sha256_file(
        path
    )


    artifact_integrity[
        name
    ] = (
        current_sha
        ==
        EXPECTED_ARTIFACT_SHA256[
            name
        ]
    )


    print(
        f"{name:26s}: "
        f"{artifact_integrity[name]}"
    )


FROZEN_ARTIFACT_INTEGRITY = all(
    artifact_integrity.values()
)


if not FROZEN_ARTIFACT_INTEGRITY:

    raise RuntimeError(
        "One or more frozen preprocessing artifacts changed."
    )


print(
    "\nFROZEN_ARTIFACT_INTEGRITY:",
    FROZEN_ARTIFACT_INTEGRITY
)


In [ ]:
# ====================================================
# 06.2 — LOAD FROZEN PREPROCESSING CONTRACT
# ====================================================

DATASET_PATH = (
    ARTIFACT_PATHS[
        "dataset"
    ]
)

METADATA_PATH = (
    ARTIFACT_PATHS[
        "metadata"
    ]
)

SAMPLE_METADATA_PATH = (
    ARTIFACT_PATHS[
        "sample_metadata"
    ]
)

SPLIT_ASSIGNMENT_PATH = (
    ARTIFACT_PATHS[
        "split_assignment"
    ]
)


dataset_metadata = json.loads(
    METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


EXPECTED_FEATURES = [
    "frequency_deviation_hz",
    "mechanical_power_kw",
    "electrical_power_kw",
    "dump_power_kw",
]


EXPECTED_TRAIN_FAMILIES = [
    "D20_L-10",
    "D30_L+10",
    "D30_L+20",
    "D30_L-10",
    "D30_L-20",
    "D40_L+20",
    "D40_L-10",
]


EXPECTED_VALIDATION_FAMILIES = [
    "D20_L-20",
    "D40_L+10",
]


EXPECTED_TEST_FAMILIES = [
    "D20_L+10",
    "D40_L-20",
]


metadata_contract_ok = all(
    [
        dataset_metadata[
            "dataset_status"
        ]
        ==
        "FINAL_LSTM_QLSTM_READY",

        dataset_metadata[
            "supervised_window_count"
        ]
        ==
        17600,

        dataset_metadata[
            "sequence_length"
        ]
        ==
        11,

        dataset_metadata[
            "feature_count"
        ]
        ==
        4,

        dataset_metadata[
            "features"
        ]
        ==
        EXPECTED_FEATURES,

        dataset_metadata[
            "observation_dt_s"
        ]
        ==
        0.05,

        dataset_metadata[
            "observation_frequency_hz"
        ]
        ==
        20.0,

        dataset_metadata[
            "window_span_s"
        ]
        ==
        0.5,

        dataset_metadata[
            "train_windows"
        ]
        ==
        11200,

        dataset_metadata[
            "validation_windows"
        ]
        ==
        3200,

        dataset_metadata[
            "test_windows"
        ]
        ==
        3200,

        dataset_metadata[
            "split_candidate_id"
        ]
        ==
        204,

        dataset_metadata[
            "train_families"
        ]
        ==
        EXPECTED_TRAIN_FAMILIES,

        dataset_metadata[
            "validation_families"
        ]
        ==
        EXPECTED_VALIDATION_FAMILIES,

        dataset_metadata[
            "test_families"
        ]
        ==
        EXPECTED_TEST_FAMILIES,

        dataset_metadata[
            "family_leakage"
        ]
        is False,

        dataset_metadata[
            "scenario_leakage"
        ]
        is False,

        dataset_metadata[
            "row_random_split_used"
        ]
        is False,

        dataset_metadata[
            "boundary_stress_test_in_supervised_tensor"
        ]
        is False,
    ]
)


if not metadata_contract_ok:

    raise RuntimeError(
        "Frozen preprocessing metadata contract changed."
    )


print(
    "Dataset status:",
    dataset_metadata[
        "dataset_status"
    ]
)

print(
    "Final windows:",
    dataset_metadata[
        "supervised_window_count"
    ]
)

print(
    "Window:",
    (
        dataset_metadata[
            "sequence_length"
        ],
        dataset_metadata[
            "window_span_s"
        ],
    )
)

print(
    "Observation cadence [s]:",
    dataset_metadata[
        "observation_dt_s"
    ]
)

print(
    "Train / Validation / Test:",
    (
        dataset_metadata[
            "train_windows"
        ],
        dataset_metadata[
            "validation_windows"
        ],
        dataset_metadata[
            "test_windows"
        ],
    )
)

print(
    "Metadata contract valid:",
    metadata_contract_ok
)


In [ ]:
# ====================================================
# 06.3 — LOAD FINAL LSTM/QLSTM TENSORS
# ====================================================

with np.load(
    DATASET_PATH,
    allow_pickle=False,
) as final_npz:

    current_schema = {
        key:
            {
                "shape":
                    list(
                        final_npz[
                            key
                        ].shape
                    ),

                "dtype":
                    str(
                        final_npz[
                            key
                        ].dtype
                    ),
            }

        for key
        in final_npz.files
    }


    if (
        current_schema
        !=
        EXPECTED_ARRAY_SCHEMA
    ):

        raise RuntimeError(
            "Frozen final tensor schema changed."
        )


    X_train = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "X_train"
            ]
        ]
        .copy()
    )


    y_train = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "y_train"
            ]
        ]
        .copy()
    )


    X_val = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "X_val"
            ]
        ]
        .copy()
    )


    y_val = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "y_val"
            ]
        ]
        .copy()
    )


    X_test = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "X_test"
            ]
        ]
        .copy()
    )


    y_test = (
        final_npz[
            CANONICAL_ARRAY_KEYS[
                "y_test"
            ]
        ]
        .copy()
    )


expected_shapes = {
    "X_train":
        (11200, 11, 4),

    "y_train":
        (11200, 2),

    "X_val":
        (3200, 11, 4),

    "y_val":
        (3200, 2),

    "X_test":
        (3200, 11, 4),

    "y_test":
        (3200, 2),
}


arrays = {
    "X_train": X_train,
    "y_train": y_train,
    "X_val": X_val,
    "y_val": y_val,
    "X_test": X_test,
    "y_test": y_test,
}


for name, array in arrays.items():

    if (
        array.shape
        !=
        expected_shapes[
            name
        ]
    ):

        raise RuntimeError(
            f"{name} shape mismatch."
        )


    if not np.isfinite(
        array
    ).all():

        raise RuntimeError(
            f"{name} contains non-finite values."
        )


    print(
        f"{name:8s}: "
        f"{array.shape} | "
        f"{array.dtype}"
    )


FINAL_TENSOR_SCHEMA_VALID = True


print(
    "\nFINAL_TENSOR_SCHEMA_VALID:",
    FINAL_TENSOR_SCHEMA_VALID
)


In [ ]:
# ====================================================
# 06.4 — FAMILY / SCENARIO LEAKAGE AUDIT
# ====================================================

sample_metadata = pd.read_csv(
    SAMPLE_METADATA_PATH
)


split_assignment = pd.read_csv(
    SPLIT_ASSIGNMENT_PATH
)


if (
    sample_metadata.columns.tolist()
    !=
    EXPECTED_SAMPLE_METADATA_COLUMNS
):

    raise RuntimeError(
        "Sample metadata schema changed."
    )


if len(
    sample_metadata
) != 17600:

    raise RuntimeError(
        "Expected 17,600 sample metadata rows."
    )


family_split_nunique = (
    split_assignment
    .groupby(
        "family_key"
    )[
        "split"
    ]
    .nunique()
)


FAMILY_LEAKAGE_FREE = bool(
    (
        family_split_nunique
        ==
        1
    ).all()
)


expected_split_sets = {
    "train":
        set(
            EXPECTED_TRAIN_FAMILIES
        ),

    "validation":
        set(
            EXPECTED_VALIDATION_FAMILIES
        ),

    "test":
        set(
            EXPECTED_TEST_FAMILIES
        ),
}


observed_split_sets = {
    split:
        set(
            split_assignment.loc[
                split_assignment[
                    "split"
                ]
                ==
                split,
                "family_key",
            ]
        )

    for split
    in (
        "train",
        "validation",
        "test",
    )
}


SPLIT_ASSIGNMENT_VALID = bool(
    observed_split_sets
    ==
    expected_split_sets
)


BOUNDARY_EXCLUDED = bool(
    "D20_L+20"
    not in
    set(
        split_assignment[
            "family_key"
        ]
    )
)


print(
    "Train families:",
    sorted(
        observed_split_sets[
            "train"
        ]
    )
)

print(
    "Validation families:",
    sorted(
        observed_split_sets[
            "validation"
        ]
    )
)

print(
    "Test families:",
    sorted(
        observed_split_sets[
            "test"
        ]
    )
)

print(
    "\nFamily leakage-free:",
    FAMILY_LEAKAGE_FREE
)

print(
    "Split assignment valid:",
    SPLIT_ASSIGNMENT_VALID
)

print(
    "D20_L+20 excluded:",
    BOUNDARY_EXCLUDED
)


if not all(
    [
        FAMILY_LEAKAGE_FREE,
        SPLIT_ASSIGNMENT_VALID,
        BOUNDARY_EXCLUDED,
    ]
):

    raise RuntimeError(
        "Frozen split contract failed."
    )


In [ ]:
# ============================================================
# 06.5 — TRAIN-ONLY SCALER AUDIT
# ============================================================

EXPECTED_FEATURE_SCALER_RECORDS = [
    {
        "feature": "frequency_deviation_hz",
        "train_mean": 3.337386389654714e-16,
        "train_std": 0.0118152103182548,
        "train_min": -0.1146640050752267,
        "train_max": 0.1146640050752054
    },
    {
        "feature": "mechanical_power_kw",
        "train_mean": 90.0,
        "train_std": 7.071067811865476,
        "train_min": 80.0,
        "train_max": 100.0
    },
    {
        "feature": "electrical_power_kw",
        "train_mean": 90.0000000000001,
        "train_std": 7.21950285455603,
        "train_min": 60.0,
        "train_max": 120.0
    },
    {
        "feature": "dump_power_kw",
        "train_mean": 31.42857142857152,
        "train_std": 14.275213338693552,
        "train_min": 4.263396671205244,
        "train_max": 55.73660332878959
    }
]

EXPECTED_TARGET_SCALER_RECORDS = [
    {
        "target": "kp_target",
        "train_mean": 2.7428571428573214,
        "train_std": 0.4027659471246905,
        "train_min": 2.16,
        "train_max": 3.36
    },
    {
        "target": "ki_target",
        "train_mean": 5.390171428572276,
        "train_std": 3.7607088380913134,
        "train_min": 3.0368,
        "train_max": 14.144
    }
]

EXPECTED_SCALERS_JSON = {
    "clipping_applied": false,
    "feature_max_train": [
        0.1146640050752054,
        100.0,
        120.0,
        55.73660332878959
    ],
    "feature_mean": [
        3.337386389654714e-16,
        90.0,
        90.0000000000001,
        31.428571428571516
    ],
    "feature_min_train": [
        -0.1146640050752267,
        80.0,
        60.0,
        4.263396671205244
    ],
    "feature_names": [
        "frequency_deviation_hz",
        "mechanical_power_kw",
        "electrical_power_kw",
        "dump_power_kw"
    ],
    "feature_scaler_fit_observation_count": 11900,
    "feature_scaler_fit_population": "UNIQUE_20HZ_OBSERVATIONS_USED_BY_TRAIN_WINDOWS",
    "feature_std": [
        0.011815210318254889,
        7.0710678118654755,
        7.21950285455603,
        14.275213338693552
    ],
    "formula": "z=(value-train_mean)/train_std",
    "inverse_formula": "value=z*train_std+train_mean",
    "scaler_type": "TRAIN_ONLY_Z_SCORE_STANDARDIZATION",
    "target_max_train": [
        3.36,
        14.144
    ],
    "target_mean": [
        2.7428571428573214,
        5.390171428572276
    ],
    "target_min_train": [
        2.16,
        3.0368
    ],
    "target_names": [
        "kp_target",
        "ki_target"
    ],
    "target_scaler_fit_population": "TRAIN_WINDOWS_ONLY",
    "target_std": [
        0.40276594712469055,
        3.7607088380913134
    ],
    "test_used_for_fit": false,
    "validation_used_for_fit": false
}


feature_scaler = pd.read_csv(
    ARTIFACT_PATHS[
        "feature_scaler"
    ]
)


target_scaler = pd.read_csv(
    ARTIFACT_PATHS[
        "target_scaler"
    ]
)


scalers_json = json.loads(
    ARTIFACT_PATHS[
        "scalers"
    ].read_text(
        encoding="utf-8"
    )
)


feature_scaler_match = bool(
    feature_scaler.to_dict(
        orient="records"
    )
    ==
    EXPECTED_FEATURE_SCALER_RECORDS
)


target_scaler_match = bool(
    target_scaler.to_dict(
        orient="records"
    )
    ==
    EXPECTED_TARGET_SCALER_RECORDS
)


scalers_json_match = bool(
    scalers_json
    ==
    EXPECTED_SCALERS_JSON
)


# Target scaling was fitted on the train-window targets.
target_mean_after_scaling = (
    y_train
    .astype(
        np.float64
    )
    .mean(
        axis=0
    )
)


target_std_after_scaling = (
    y_train
    .astype(
        np.float64
    )
    .std(
        axis=0,
        ddof=0,
    )
)


target_scaling_sanity = bool(
    np.all(
        np.abs(
            target_mean_after_scaling
        )
        <
        1e-5
    )
    and
    np.allclose(
        target_std_after_scaling,
        np.ones(
            2
        ),
        rtol=0.0,
        atol=1e-5,
    )
)


TRAIN_ONLY_SCALER_CONTRACT_VALID = all(
    [
        feature_scaler_match,
        target_scaler_match,
        scalers_json_match,
        target_scaling_sanity,

        dataset_metadata[
            "feature_standardization"
        ]
        ==
        "TRAIN_ONLY_UNIQUE_OBSERVATION_Z_SCORE",

        dataset_metadata[
            "target_standardization"
        ]
        ==
        "TRAIN_ONLY_WINDOW_TARGET_Z_SCORE",
    ]
)


print(
    "Feature scaler artifact exact:",
    feature_scaler_match
)

print(
    "Target scaler artifact exact:",
    target_scaler_match
)

print(
    "Scaler JSON exact:",
    scalers_json_match
)

print(
    "Scaled y_train mean:",
    target_mean_after_scaling
)

print(
    "Scaled y_train std:",
    target_std_after_scaling
)

print(
    "\nTRAIN_ONLY_SCALER_CONTRACT_VALID:",
    TRAIN_ONLY_SCALER_CONTRACT_VALID
)


if not TRAIN_ONLY_SCALER_CONTRACT_VALID:

    raise RuntimeError(
        "Frozen scaler contract failed."
    )


## 3. Mengapa Window Harus Berakhir Setelah Gangguan?

Window pre-disturbance tidak digunakan sebagai target supervised utama
karena kondisi sebelum gangguan belum memuat informasi temporal yang
cukup untuk membedakan kebutuhan gain dari beberapa keluarga dinamik.

Dengan observation cadence 0,05 s dan panjang sequence 11:

- skenario dengan gangguan 1,5 s menghasilkan **170** window eligible;
- skenario dengan gangguan 2,5 s menghasilkan **150** window eligible.

Untuk 110 skenario supervised:

\[
55(170)+55(150)=17600
\]

sehingga jumlah tersebut dapat diturunkan langsung dari aturan
kausal, bukan dari random sampling.


In [ ]:
# ====================================================
# 06.6 — CAUSAL WINDOW COUNT DERIVATION
# ====================================================

OBSERVATION_DT_S = 0.05

SEQUENCE_LENGTH = 11

WINDOW_SPAN_S = (
    (
        SEQUENCE_LENGTH
        -
        1
    )
    *
    OBSERVATION_DT_S
)


def eligible_window_count(
    disturbance_time_s,
    simulation_end_s=10.0,
):

    observation_times = np.arange(
        0.0,
        simulation_end_s
        +
        OBSERVATION_DT_S
        /
        2.0,
        OBSERVATION_DT_S,
    )


    count = 0


    for end_index in range(
        SEQUENCE_LENGTH - 1,
        len(
            observation_times
        ),
    ):

        end_time = (
            observation_times[
                end_index
            ]
        )


        if (
            end_time
            >
            disturbance_time_s
        ):

            count += 1


    return count


windows_td_15 = (
    eligible_window_count(
        1.5
    )
)


windows_td_25 = (
    eligible_window_count(
        2.5
    )
)


derived_total = (
    55
    *
    windows_td_15
    +
    55
    *
    windows_td_25
)


CAUSAL_WINDOW_COUNT_VALID = bool(
    windows_td_15
    ==
    170

    and

    windows_td_25
    ==
    150

    and

    derived_total
    ==
    17600

    and

    abs(
        WINDOW_SPAN_S
        -
        0.5
    )
    <
    1e-12
)


print(
    "Window span [s]:",
    WINDOW_SPAN_S
)

print(
    "Eligible windows @ td=1.5 s:",
    windows_td_15
)

print(
    "Eligible windows @ td=2.5 s:",
    windows_td_25
)

print(
    "Derived supervised windows:",
    derived_total
)

print(
    "CAUSAL_WINDOW_COUNT_VALID:",
    CAUSAL_WINDOW_COUNT_VALID
)


if not CAUSAL_WINDOW_COUNT_VALID:

    raise RuntimeError(
        "Causal-window count contract changed."
    )


In [ ]:
# ====================================================
# 06.7 — OPTIONAL PREPROCESSING REPRODUCTION FUNCTIONS
# ====================================================
#
# These functions are defined but are NOT executed
# unless REBUILD_PREPROCESSING=True in the next cell.
# ====================================================

RAW_TRAJECTORY_PATH = (
    PROJECT_ROOT
    /
    "data"
    /
    "raw"
    /
    "notebook05_raw_temporal_trajectories.csv.gz"
)


REPRODUCTION_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "reproduction"
)


FEATURE_COLUMNS = [
    "frequency_deviation_hz",
    "mechanical_power_kw",
    "electrical_power_kw",
    "dump_power_kw",
]


TARGET_COLUMNS = [
    "kp_target",
    "ki_target",
]


def resample_trajectory_20hz(
    scenario_df,
):

    scenario_df = (
        scenario_df
        .sort_values(
            "time_s"
        )
        .reset_index(
            drop=True
        )
    )


    # Notebook 05 reproduction uses dt=0.0025 s.
    expected_raw_count = 4001


    if len(
        scenario_df
    ) != expected_raw_count:

        raise RuntimeError(
            "Expected 4001 RK4 samples per scenario."
        )


    observation_df = (
        scenario_df
        .iloc[
            ::20
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    if len(
        observation_df
    ) != 201:

        raise RuntimeError(
            "Expected 201 observations at 20 Hz."
        )


    expected_times = np.arange(
        0.0,
        10.0 + 0.025,
        0.05,
    )


    if not np.allclose(
        observation_df[
            "time_s"
        ].to_numpy(
            dtype=float
        ),
        expected_times,
        rtol=0.0,
        atol=1e-10,
    ):

        raise RuntimeError(
            "20-Hz observation grid mismatch."
        )


    return observation_df


def build_causal_windows(
    observation_df,
):

    disturbance_time = float(
        observation_df[
            "disturbance_time_s"
        ].iloc[0]
    )


    scenario_id = str(
        observation_df[
            "scenario_id"
        ].iloc[0]
    )


    family_key = str(
        observation_df[
            "family_key"
        ].iloc[0]
    )


    X_rows = []

    y_rows = []

    metadata_rows = []

    used_observation_indices = set()


    features = (
        observation_df[
            FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    target = (
        observation_df[
            TARGET_COLUMNS
        ]
        .iloc[0]
        .to_numpy(
            dtype=np.float64
        )
    )


    for end_index in range(
        SEQUENCE_LENGTH - 1,
        len(
            observation_df
        ),
    ):

        end_time = float(
            observation_df[
                "time_s"
            ].iloc[
                end_index
            ]
        )


        if not (
            end_time
            >
            disturbance_time
        ):

            continue


        start_index = (
            end_index
            -
            (
                SEQUENCE_LENGTH
                -
                1
            )
        )


        X_rows.append(
            features[
                start_index:
                end_index + 1
            ]
        )


        y_rows.append(
            target.copy()
        )


        for obs_index in range(
            start_index,
            end_index + 1,
        ):

            used_observation_indices.add(
                obs_index
            )


        metadata_rows.append(
            {
                "scenario_id":
                    scenario_id,

                "family_key":
                    family_key,

                "window_start_time_s":
                    float(
                        observation_df[
                            "time_s"
                        ].iloc[
                            start_index
                        ]
                    ),

                "window_end_time_s":
                    end_time,

                "disturbance_time_s":
                    disturbance_time,
            }
        )


    return (
        X_rows,
        y_rows,
        metadata_rows,
        sorted(
            used_observation_indices
        ),
    )


print(
    "Optional preprocessing functions defined:",
    True
)


In [ ]:
# ====================================================
# 06.8 — OPTIONAL FULL PREPROCESSING REBUILD
# ====================================================
#
# SAFE DEFAULT:
#
# No scientific artifact is rebuilt.
# ====================================================

REBUILD_PREPROCESSING = False


PREPROCESSING_REBUILD_PERFORMED = False


if not REBUILD_PREPROCESSING:

    print(
        "REBUILD_PREPROCESSING = False"
    )

    print(
        "Frozen final tensors were verified only."
    )

    print(
        "No preprocessing artifact was overwritten."
    )


else:

    if not RAW_TRAJECTORY_PATH.exists():

        raise RuntimeError(
            "Raw Notebook-05 trajectory file is missing. "
            "Run Notebook 05 explicitly with "
            "REPRODUCE_RAW_TRAJECTORIES=True first."
        )


    raw = pd.read_csv(
        RAW_TRAJECTORY_PATH
    )


    required_raw_columns = {
        "scenario_id",
        "family_key",
        "time_s",
        "disturbance_time_s",
        "frequency_deviation_hz",
        "mechanical_power_kw",
        "electrical_power_kw",
        "dump_power_kw",
        "kp_target",
        "ki_target",
        "supervised_eligible",
    }


    if not required_raw_columns.issubset(
        raw.columns
    ):

        raise RuntimeError(
            "Raw Notebook-05 schema is incomplete."
        )


    raw_supervised = (
        raw.loc[
            raw[
                "supervised_eligible"
            ].astype(
                bool
            )
        ]
        .copy()
    )


    if (
        raw_supervised[
            "scenario_id"
        ].nunique()
        !=
        110
    ):

        raise RuntimeError(
            "Expected 110 supervised scenarios."
        )


    split_lookup = (
        split_assignment[
            [
                "family_key",
                "split",
            ]
        ]
        .set_index(
            "family_key"
        )[
            "split"
        ]
        .to_dict()
    )


    all_X = []

    all_y = []

    all_metadata = []

    train_unique_observations = []


    for scenario_id, scenario_df in (
        raw_supervised
        .groupby(
            "scenario_id",
            sort=True,
        )
    ):

        obs = (
            resample_trajectory_20hz(
                scenario_df
            )
        )


        (
            X_rows,
            y_rows,
            metadata_rows,
            used_indices,
        ) = build_causal_windows(
            obs
        )


        family = str(
            obs[
                "family_key"
            ].iloc[0]
        )


        split = (
            split_lookup[
                family
            ]
        )


        for metadata_row in metadata_rows:

            metadata_row[
                "split"
            ] = split


        all_X.extend(
            X_rows
        )

        all_y.extend(
            y_rows
        )

        all_metadata.extend(
            metadata_rows
        )


        # Feature scaler:
        # unique physical observations used by
        # TRAIN windows only.
        if split == "train":

            train_unique_observations.append(
                obs.iloc[
                    used_indices
                ][
                    FEATURE_COLUMNS
                ].to_numpy(
                    dtype=np.float64
                )
            )


    X_raw = np.stack(
        all_X
    )


    y_raw = np.stack(
        all_y
    )


    rebuild_meta = pd.DataFrame(
        all_metadata
    )


    if len(
        X_raw
    ) != 17600:

        raise RuntimeError(
            "Rebuild did not produce 17,600 windows."
        )


    train_mask = (
        rebuild_meta[
            "split"
        ]
        ==
        "train"
    ).to_numpy()


    val_mask = (
        rebuild_meta[
            "split"
        ]
        ==
        "validation"
    ).to_numpy()


    test_mask = (
        rebuild_meta[
            "split"
        ]
        ==
        "test"
    ).to_numpy()


    if (
        int(
            train_mask.sum()
        ),
        int(
            val_mask.sum()
        ),
        int(
            test_mask.sum()
        ),
    ) != (
        11200,
        3200,
        3200,
    ):

        raise RuntimeError(
            "Rebuilt split sizes do not match "
            "the frozen contract."
        )


    train_feature_observations = np.concatenate(
        train_unique_observations,
        axis=0,
    )


    if (
        len(
            train_feature_observations
        )
        !=
        11900
    ):

        raise RuntimeError(
            "Expected 11,900 unique training "
            "observation points for feature scaler."
        )


    feature_mean = (
        train_feature_observations
        .mean(
            axis=0
        )
    )


    feature_std = (
        train_feature_observations
        .std(
            axis=0,
            ddof=0,
        )
    )


    target_mean = (
        y_raw[
            train_mask
        ]
        .mean(
            axis=0
        )
    )


    target_std = (
        y_raw[
            train_mask
        ]
        .std(
            axis=0,
            ddof=0,
        )
    )


    if np.any(
        feature_std
        <=
        0.0
    ):

        raise RuntimeError(
            "Zero feature standard deviation."
        )


    if np.any(
        target_std
        <=
        0.0
    ):

        raise RuntimeError(
            "Zero target standard deviation."
        )


    X_scaled = (
        (
            X_raw
            -
            feature_mean[
                None,
                None,
                :
            ]
        )
        /
        feature_std[
            None,
            None,
            :
        ]
    )


    y_scaled = (
        (
            y_raw
            -
            target_mean[
                None,
                :
            ]
        )
        /
        target_std[
            None,
            :
        ]
    )


    rebuilt_arrays = {
        CANONICAL_ARRAY_KEYS[
            "X_train"
        ]:
            X_scaled[
                train_mask
            ].astype(
                np.float32
            ),

        CANONICAL_ARRAY_KEYS[
            "y_train"
        ]:
            y_scaled[
                train_mask
            ].astype(
                np.float32
            ),

        CANONICAL_ARRAY_KEYS[
            "X_val"
        ]:
            X_scaled[
                val_mask
            ].astype(
                np.float32
            ),

        CANONICAL_ARRAY_KEYS[
            "y_val"
        ]:
            y_scaled[
                val_mask
            ].astype(
                np.float32
            ),

        CANONICAL_ARRAY_KEYS[
            "X_test"
        ]:
            X_scaled[
                test_mask
            ].astype(
                np.float32
            ),

        CANONICAL_ARRAY_KEYS[
            "y_test"
        ]:
            y_scaled[
                test_mask
            ].astype(
                np.float32
            ),
    }


    rebuilt_schema = {
        key:
            {
                "shape":
                    list(
                        value.shape
                    ),

                "dtype":
                    str(
                        value.dtype
                    ),
            }

        for key, value
        in rebuilt_arrays.items()
    }


    # Scientific schema must match.
    schema_match = all(
        rebuilt_schema[
            key
        ][
            "shape"
        ]
        ==
        EXPECTED_ARRAY_SCHEMA[
            key
        ][
            "shape"
        ]

        for key
        in rebuilt_schema
    )


    if not schema_match:

        raise RuntimeError(
            "Rebuilt tensor schema differs "
            "from frozen final dataset."
        )


    REPRODUCTION_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    rebuilt_npz_path = (
        REPRODUCTION_DIR
        /
        "notebook06_rebuilt_dataset.npz"
    )


    rebuilt_meta_path = (
        REPRODUCTION_DIR
        /
        "notebook06_rebuilt_window_metadata.csv.gz"
    )


    rebuilt_scaler_path = (
        REPRODUCTION_DIR
        /
        "notebook06_rebuilt_scaler_summary.json"
    )


    np.savez_compressed(
        rebuilt_npz_path,
        **rebuilt_arrays,
    )


    rebuild_meta.to_csv(
        rebuilt_meta_path,
        index=False,
        compression="gzip",
    )


    rebuilt_scaler_path.write_text(
        json.dumps(
            {
                "feature_mean":
                    feature_mean.tolist(),

                "feature_std":
                    feature_std.tolist(),

                "target_mean":
                    target_mean.tolist(),

                "target_std":
                    target_std.tolist(),

                "unique_train_observation_count":
                    int(
                        len(
                            train_feature_observations
                        )
                    ),

                "frozen_final_files_overwritten":
                    False,
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )


    PREPROCESSING_REBUILD_PERFORMED = True


    print(
        "Rebuilt dataset:",
        rebuilt_npz_path
    )

    print(
        "Rebuilt metadata:",
        rebuilt_meta_path
    )

    print(
        "Rebuilt scaler summary:",
        rebuilt_scaler_path
    )

    print(
        "Frozen data/final overwritten: False"
    )


## 4. Handoff ke `07_train_lstm.ipynb`

Artefak otoritatif yang diteruskan ke tahap pelatihan adalah:

`data/final/qlstm_lstm_dataset_final.npz`

dengan:

- `X_train`: 11.200 × 11 × 4;
- `y_train`: 11.200 × 2;
- `X_val`: 3.200 × 11 × 4;
- `y_val`: 3.200 × 2;
- `X_test`: 3.200 × 11 × 4;
- `y_test`: 3.200 × 2.

Dataset yang sama dipakai sebagai dasar LSTM maupun QLSTM.

Tidak ada dataset baru yang dipilih setelah melihat performa model.


In [ ]:
# ====================================================
# 06.9 — PREPROCESSING READINESS SUMMARY
# ====================================================

NOTEBOOK_06_PREPROCESSING_CONTRACT_READY = all(
    [
        FROZEN_ARTIFACT_INTEGRITY,
        metadata_contract_ok,
        FINAL_TENSOR_SCHEMA_VALID,
        FAMILY_LEAKAGE_FREE,
        SPLIT_ASSIGNMENT_VALID,
        BOUNDARY_EXCLUDED,
        TRAIN_ONLY_SCALER_CONTRACT_VALID,
        CAUSAL_WINDOW_COUNT_VALID,
    ]
)


print("=" * 72)
print("06_preprocessing.ipynb — SUMMARY")
print("=" * 72)


print(
    "Dataset status                      :",
    dataset_metadata[
        "dataset_status"
    ]
)

print(
    "Observation cadence [Hz]            :",
    dataset_metadata[
        "observation_frequency_hz"
    ]
)

print(
    "Sequence length                     :",
    dataset_metadata[
        "sequence_length"
    ]
)

print(
    "Window span [s]                     :",
    dataset_metadata[
        "window_span_s"
    ]
)

print(
    "Feature count                       :",
    dataset_metadata[
        "feature_count"
    ]
)

print(
    "Total supervised windows            :",
    dataset_metadata[
        "supervised_window_count"
    ]
)

print(
    "Train windows                       :",
    len(
        X_train
    )
)

print(
    "Validation windows                  :",
    len(
        X_val
    )
)

print(
    "Test windows                        :",
    len(
        X_test
    )
)

print(
    "Family leakage                      : False"
)

print(
    "Scenario leakage                    : False"
)

print(
    "Row-random split                    : False"
)

print(
    "Boundary D20_L+20 in tensor         : False"
)

print(
    "Train-only scaler                   : True"
)

print(
    "Preprocessing rebuild requested     :",
    REBUILD_PREPROCESSING
)

print(
    "Preprocessing rebuild performed     :",
    PREPROCESSING_REBUILD_PERFORMED
)

print(
    "NOTEBOOK 06 PREPROCESSING READY     :",
    NOTEBOOK_06_PREPROCESSING_CONTRACT_READY
)


if NOTEBOOK_06_PREPROCESSING_CONTRACT_READY:

    print(
        "\nNEXT NOTEBOOK:"
    )

    print(
        "07_train_lstm.ipynb"
    )
